### A Holistic Utility-Based Modeling Framework for Quality--Efficiency Trade-offs in Edge Language Models
#### Partha Pratim Ray, Sikkim University, parthapratimray1986@gmail.com

In [4]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# Configuration
# =========================================================
INPUT_CSV = "edge_llm_results.csv"
OUTDIR = "paper_outputs"
os.makedirs(OUTDIR, exist_ok=True)

# Scenario weights: alpha(Q), beta(L), gamma(E), delta(R), eta(T)
SCENARIOS = {
    "balanced":         {"alpha": 0.30, "beta": 0.20, "gamma": 0.20, "delta": 0.15, "eta": 0.15},
    "latency_critical": {"alpha": 0.15, "beta": 0.45, "gamma": 0.15, "delta": 0.10, "eta": 0.15},
    "energy_critical":  {"alpha": 0.15, "beta": 0.15, "gamma": 0.45, "delta": 0.10, "eta": 0.15},
    "quality_critical": {"alpha": 0.55, "beta": 0.15, "gamma": 0.10, "delta": 0.10, "eta": 0.10},
    "memory_critical":  {"alpha": 0.15, "beta": 0.15, "gamma": 0.10, "delta": 0.45, "eta": 0.15},
    "thermal_critical": {"alpha": 0.15, "beta": 0.15, "gamma": 0.15, "delta": 0.10, "eta": 0.45},
}

EPS = 1e-9

# =========================================================
# Helpers
# =========================================================
def minmax_normalize(series: pd.Series) -> pd.Series:
    smin = series.min()
    smax = series.max()
    if abs(smax - smin) < EPS:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - smin) / (smax - smin)

def invert_cost(series: pd.Series) -> pd.Series:
    # Smaller-is-better cost -> larger-is-better score
    return 1.0 - minmax_normalize(series)

def compute_pareto_flags(df: pd.DataFrame, benefit_cols=None, cost_cols=None) -> pd.Series:
    """
    Returns True for Pareto-optimal rows.
    benefit_cols: larger is better
    cost_cols: smaller is better
    """
    if benefit_cols is None:
        benefit_cols = []
    if cost_cols is None:
        cost_cols = []

    n = len(df)
    pareto = np.ones(n, dtype=bool)

    for i in range(n):
        for j in range(n):
            if i == j:
                continue

            no_worse = True
            strictly_better = False

            for col in benefit_cols:
                if df.iloc[j][col] < df.iloc[i][col]:
                    no_worse = False
                    break
                if df.iloc[j][col] > df.iloc[i][col]:
                    strictly_better = True

            if not no_worse:
                continue

            for col in cost_cols:
                if df.iloc[j][col] > df.iloc[i][col]:
                    no_worse = False
                    break
                if df.iloc[j][col] < df.iloc[i][col]:
                    strictly_better = True

            if no_worse and strictly_better:
                pareto[i] = False
                break

    return pd.Series(pareto, index=df.index)

def save_table_as_latex(df: pd.DataFrame, path: str, caption: str, label: str, float_format="%.4f"):
    latex = df.to_latex(
        index=False,
        escape=False,
        float_format=lambda x: float_format % x if isinstance(x, (int, float, np.floating)) else str(x),
        caption=caption,
        label=label
    )
    with open(path, "w", encoding="utf-8") as f:
        f.write(latex)

def annotate_bars(ax, rects, fmt="{:.3f}", fontsize=7):
    for rect in rects:
        height = rect.get_height()
        va = "bottom" if height >= 0 else "top"
        offset = 2 if height >= 0 else -2
        ax.annotate(
            fmt.format(height),
            xy=(rect.get_x() + rect.get_width() / 2, height),
            xytext=(0, offset),
            textcoords="offset points",
            ha="center",
            va=va,
            fontsize=fontsize
        )

# =========================================================
# Load data
# =========================================================
df = pd.read_csv(INPUT_CSV)

required_cols = [
    "model", "quality", "latency_s", "energy_j", "memory_mb", "thermal_c",
    "tokens_per_sec", "power_w"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# =========================================================
# Normalized metrics
# Quality is benefit; latency/energy/memory/thermal are costs
# =========================================================
df["Q_norm"] = minmax_normalize(df["quality"])
df["L_norm"] = minmax_normalize(df["latency_s"])
df["E_norm"] = minmax_normalize(df["energy_j"])
df["R_norm"] = minmax_normalize(df["memory_mb"])
df["T_norm"] = minmax_normalize(df["thermal_c"])

df["Q_score"] = df["Q_norm"]
df["L_score"] = invert_cost(df["latency_s"])
df["E_score"] = invert_cost(df["energy_j"])
df["R_score"] = invert_cost(df["memory_mb"])
df["T_score"] = invert_cost(df["thermal_c"])

# =========================================================
# Composite efficiency metrics
# =========================================================
df["QEE"] = df["quality"] / (df["energy_j"] + EPS)
df["QLE"] = df["quality"] / (df["latency_s"] + EPS)
df["QRE"] = df["quality"] / (
    (df["energy_j"] + EPS) ** 0.4 *
    (df["latency_s"] + EPS) ** 0.3 *
    (df["memory_mb"] + EPS) ** 0.3
)

df["QEE_log"] = np.log(df["QEE"] + EPS)
df["QLE_log"] = np.log(df["QLE"] + EPS)

# =========================================================
# Scenario-based utility
# U = alpha*Q_norm - beta*L_norm - gamma*E_norm - delta*R_norm - eta*T_norm
# =========================================================
for scenario, w in SCENARIOS.items():
    df[f"utility_{scenario}"] = (
        w["alpha"] * df["Q_norm"]
        - w["beta"] * df["L_norm"]
        - w["gamma"] * df["E_norm"]
        - w["delta"] * df["R_norm"]
        - w["eta"] * df["T_norm"]
    )

# Risk-sensitive utility example using variance proxy from normalized costs
# If you later have repeated runs, replace with actual utility variance.
df["risk_proxy"] = df[["L_norm", "E_norm", "R_norm", "T_norm"]].var(axis=1)
lambda_risk = 0.15
df["utility_balanced_risk"] = df["utility_balanced"] - lambda_risk * df["risk_proxy"]

# =========================================================
# Pareto optimality
# Benefit: quality
# Costs: latency, energy, memory, thermal
# =========================================================
df["pareto_optimal"] = compute_pareto_flags(
    df,
    benefit_cols=["quality"],
    cost_cols=["latency_s", "energy_j", "memory_mb", "thermal_c"]
)

# =========================================================
# Rank summary
# =========================================================
rank_cols = ["utility_balanced", "utility_latency_critical", "utility_energy_critical",
             "utility_quality_critical", "QEE", "QLE", "QRE"]

summary_cols = [
    "model", "quality", "latency_s", "energy_j", "memory_mb", "thermal_c",
    "tokens_per_sec", "power_w", "QEE", "QLE", "QRE",
    "utility_balanced", "utility_balanced_risk", "pareto_optimal"
]

summary_df = df[summary_cols].copy()
summary_df = summary_df.sort_values(by="utility_balanced", ascending=False)
summary_df.to_csv(os.path.join(OUTDIR, "model_summary.csv"), index=False)

save_table_as_latex(
    summary_df.round(4),
    os.path.join(OUTDIR, "table_model_summary.tex"),
    caption="Holistic model summary under the proposed utility-based evaluation framework.",
    label="tab:holistic_model_summary"
)

scenario_winners = []
for scenario in SCENARIOS:
    idx = df[f"utility_{scenario}"].idxmax()
    scenario_winners.append({
        "scenario": scenario,
        "best_model": df.loc[idx, "model"],
        "utility": df.loc[idx, f"utility_{scenario}"]
    })

scenario_df = pd.DataFrame(scenario_winners)
scenario_df.to_csv(os.path.join(OUTDIR, "scenario_winners.csv"), index=False)

save_table_as_latex(
    scenario_df.round(4),
    os.path.join(OUTDIR, "table_scenario_winners.tex"),
    caption="Best-performing model under each deployment regime.",
    label="tab:scenario_winners"
)

# =========================================================
# Figure 1: Holistic normalized metric profile
# =========================================================
plot_df = df[["model", "Q_norm", "L_score", "E_score", "R_score", "T_score"]].copy()
plot_df = plot_df.set_index("model")
plot_df.to_csv(os.path.join(OUTDIR, "fig1_holistic_profile.csv"))

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(plot_df.index))
width = 0.14

metric_names = ["Q_norm", "L_score", "E_score", "R_score", "T_score"]
for i, metric in enumerate(metric_names):
    bars = ax.bar(x + (i - 2) * width, plot_df[metric].values, width=width, label=metric)
    annotate_bars(ax, bars, fmt="{:.3f}", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(plot_df.index, rotation=0)
ax.set_ylabel("Normalized score")
ax.set_title("Holistic normalized performance profile across models")
ax.set_ylim(0, 1.10)
ax.legend(frameon=False)
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, "fig_holistic_profile.png"), dpi=600, bbox_inches="tight")
fig.savefig(os.path.join(OUTDIR, "fig_holistic_profile.pdf"), bbox_inches="tight")
plt.close(fig)

# =========================================================
# Figure 2: Pareto frontier (Quality vs Energy)
# marker size = inverse latency score
# =========================================================
pareto_plot_df = df[["model", "quality", "energy_j", "latency_s", "L_score", "pareto_optimal"]].copy()
pareto_plot_df.to_csv(os.path.join(OUTDIR, "fig2_pareto_data.csv"), index=False)

fig, ax = plt.subplots(figsize=(7, 5))
sizes = 250 * (df["L_score"] + 0.2)

for _, row in df.iterrows():
    ax.scatter(row["energy_j"], row["quality"], s=float(sizes.loc[row.name]))
    ax.annotate(
        row["model"],
        (row["energy_j"], row["quality"]),
        fontsize=9,
        xytext=(4, 16),
        textcoords="offset points"
    )
    ax.text(
        row["energy_j"], row["quality"],
        f"({row['energy_j']:.2f}, {row['quality']:.4f})",
        fontsize=7,
        ha="left",
        va="bottom"
    )

pareto_df = df[df["pareto_optimal"]].sort_values("energy_j")
if len(pareto_df) >= 2:
    ax.plot(pareto_df["energy_j"], pareto_df["quality"])

ax.set_xlabel("Energy (J)")
ax.set_ylabel("Quality")
ax.set_title("Pareto frontier: quality versus energy")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, "fig_pareto_quality_energy.png"), dpi=600, bbox_inches="tight")
fig.savefig(os.path.join(OUTDIR, "fig_pareto_quality_energy.pdf"), bbox_inches="tight")
plt.close(fig)

# =========================================================
# Figure 3: Quality-energy-latency efficiency comparison
# =========================================================
eff_df = df[["model", "QEE", "QLE", "QRE"]].set_index("model")
eff_df.to_csv(os.path.join(OUTDIR, "fig3_efficiency.csv"))

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(eff_df.index))
width = 0.22
for i, metric in enumerate(["QEE", "QLE", "QRE"]):
    bars = ax.bar(x + (i - 1) * width, eff_df[metric].values, width=width, label=metric)
    annotate_bars(ax, bars, fmt="{:.4f}", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(eff_df.index)
ax.set_ylabel("Efficiency value")
ax.set_title("Composite efficiency metrics across candidate models")
ax.legend(frameon=False)
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, "fig_composite_efficiency.png"), dpi=600, bbox_inches="tight")
fig.savefig(os.path.join(OUTDIR, "fig_composite_efficiency.pdf"), bbox_inches="tight")
plt.close(fig)

# =========================================================
# Figure 4: Scenario-wise utility comparison
# =========================================================
utility_cols = [f"utility_{k}" for k in SCENARIOS.keys()]
utility_labels = list(SCENARIOS.keys())
df[["model"] + utility_cols].to_csv(os.path.join(OUTDIR, "fig4_scenario_utilities.csv"), index=False)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(df["model"]))
width = 0.12

for i, (col, lbl) in enumerate(zip(utility_cols, utility_labels)):
    bars = ax.bar(x + (i - (len(utility_cols)-1)/2) * width, df[col].values, width=width, label=lbl)
    annotate_bars(ax, bars, fmt="{:.2f}", fontsize=6)

ax.set_xticks(x)
ax.set_xticklabels(df["model"])
ax.set_ylabel("Utility")
ax.set_title("Scenario-wise utility under deployment priorities")
ax.legend(frameon=False, ncol=2)
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, "fig_scenario_utilities.png"), dpi=600, bbox_inches="tight")
fig.savefig(os.path.join(OUTDIR, "fig_scenario_utilities.pdf"), bbox_inches="tight")
plt.close(fig)

# =========================================================
# Figure 5: Thermal vs power with latency annotations
# =========================================================
thermal_power_df = df[["model", "power_w", "thermal_c", "latency_s"]].copy()
thermal_power_df.to_csv(os.path.join(OUTDIR, "fig5_thermal_power.csv"), index=False)

fig, ax = plt.subplots(figsize=(7, 5))
for _, row in df.iterrows():
    ax.scatter(row["power_w"], row["thermal_c"], s=220)
    label = f"{row['model']}\nL={row['latency_s']:.1f}s"
    ax.annotate(label, (row["power_w"], row["thermal_c"]), fontsize=8, xytext=(4, 18), textcoords="offset points")
    ax.text(
        row["power_w"], row["thermal_c"],
        f"({row['power_w']:.2f}, {row['thermal_c']:.2f})",
        fontsize=7,
        ha="left",
        va="bottom"
    )

ax.set_xlabel("Average power (W)")
ax.set_ylabel("Thermal steady state / observed temperature (°C)")
ax.set_title("Thermal-power operating region of edge LLMs")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, "fig_thermal_power_region.png"), dpi=600, bbox_inches="tight")
fig.savefig(os.path.join(OUTDIR, "fig_thermal_power_region.pdf"), bbox_inches="tight")
plt.close(fig)

# =========================================================
# Figure 6: Correlation matrix
# =========================================================
corr_cols = ["quality", "latency_s", "energy_j", "memory_mb", "thermal_c", "tokens_per_sec", "power_w", "QEE", "QLE", "QRE"]
corr = df[corr_cols].corr()
corr.to_csv(os.path.join(OUTDIR, "fig6_correlation_matrix.csv"))

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr.values, aspect="auto")
ax.set_xticks(np.arange(len(corr_cols)))
ax.set_yticks(np.arange(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticklabels(corr_cols)
ax.set_title("Correlation structure of quality, resource, and efficiency variables")

for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, "fig_correlation_matrix.png"), dpi=600, bbox_inches="tight")
fig.savefig(os.path.join(OUTDIR, "fig_correlation_matrix.pdf"), bbox_inches="tight")
plt.close(fig)

# =========================================================
# Figure 7: Risk-sensitive utility ranking
# =========================================================
risk_df = df[["model", "utility_balanced", "utility_balanced_risk", "risk_proxy"]].copy()
risk_df = risk_df.sort_values("utility_balanced_risk", ascending=False)
risk_df.to_csv(os.path.join(OUTDIR, "fig7_risk_utility.csv"), index=False)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(risk_df))
width = 0.35
bars1 = ax.bar(x - width / 2, risk_df["utility_balanced"].values, width=width, label="Utility")
bars2 = ax.bar(x + width / 2, risk_df["utility_balanced_risk"].values, width=width, label="Risk-sensitive utility")
annotate_bars(ax, bars1, fmt="{:.3f}", fontsize=7)
annotate_bars(ax, bars2, fmt="{:.3f}", fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels(risk_df["model"])
ax.set_ylabel("Score")
ax.set_title("Balanced utility versus risk-sensitive utility")
ax.legend(frameon=False)
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, "fig_risk_sensitive_utility.png"), dpi=600, bbox_inches="tight")
fig.savefig(os.path.join(OUTDIR, "fig_risk_sensitive_utility.pdf"), bbox_inches="tight")
plt.close(fig)

# =========================================================
# Figure 8: LaTeX-ready ranking table for main paper
# =========================================================
main_table = df[[
    "model", "quality", "latency_s", "energy_j", "memory_mb", "thermal_c",
    "QEE", "QLE", "utility_balanced", "pareto_optimal"
]].copy()

main_table = main_table.sort_values("utility_balanced", ascending=False)
save_table_as_latex(
    main_table.round(4),
    os.path.join(OUTDIR, "table_main_ranking.tex"),
    caption="Holistic ranking of candidate models under the balanced deployment regime.",
    label="tab:main_ranking"
)

# =========================================================
# Optional console summary
# =========================================================
print("\nTop models by balanced utility:")
print(df.sort_values("utility_balanced", ascending=False)[["model", "utility_balanced"]].to_string(index=False))

print("\nPareto-optimal models:")
print(df.loc[df["pareto_optimal"], ["model", "quality", "latency_s", "energy_j", "memory_mb", "thermal_c"]].to_string(index=False))

print("\n--- Figure Data Tables ---")

print("\nFig1: Holistic Profile")
print(plot_df.to_string())

print("\nFig2: Pareto Data")
print(pareto_plot_df.to_string(index=False))

print("\nFig3: Efficiency Metrics")
print(eff_df.to_string())

print("\nFig4: Scenario Utilities")
print(df[["model"] + utility_cols].to_string(index=False))

print("\nFig5: Thermal-Power Data")
print(thermal_power_df.to_string(index=False))

print("\nFig6: Correlation Matrix")
print(corr.to_string())

print("\nFig7: Risk Utility")
print(risk_df.to_string(index=False))

print(f"\nOutputs saved to: {OUTDIR}")


Top models by balanced utility:
   model  utility_balanced
 Qwen2.5          0.189843
Llama3.2          0.020472
Granite3         -0.027061
 SmolLM2         -0.700000

Pareto-optimal models:
   model  quality  latency_s  energy_j  memory_mb  thermal_c
 Qwen2.5     0.86      18.50     92.50       2100       61.2
Llama3.2     0.81      15.96     87.78       2280       63.5
Granite3     0.83      33.34     96.69       2050       60.1

--- Figure Data Tables ---

Fig1: Holistic Profile
            Q_norm   L_score   E_score   R_score   T_score
model                                                     
Qwen2.5   1.000000  0.876098  0.781481  0.886364  0.835821
Llama3.2  0.583333  1.000000  1.000000  0.477273  0.492537
SmolLM2   0.000000  0.000000  0.000000  0.000000  0.000000
Granite3  0.750000  0.152195  0.587500  1.000000  1.000000

Fig2: Pareto Data
   model  quality  energy_j  latency_s  L_score  pareto_optimal
 Qwen2.5     0.86     92.50      18.50 0.876098            True
Llama3.2   